In [61]:
import os
from dotenv import load_dotenv
load_dotenv()

import pandas as pd
import mlflow
import mlflow.catboost
from catboost import CatBoostClassifier
import psycopg
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder, 
    SplineTransformer, 
    QuantileTransformer, 
    RobustScaler,
    PolynomialFeatures,
    KBinsDiscretizer,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_auc_score, precision_score, recall_score, f1_score, log_loss


TABLE_NAME = 'clean_users_churn' # таблица с данными

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

EXPERIMENT_NAME = 'churn_laptev_ilya_sergeevich_2' # название эксперимента
RUN_NAME = "preprocessing" 
REGISTRY_MODEL_NAME = 'churn_model_laptev_ilya_sergeevich_2_b2c' # название зарегистрированной модели 

In [33]:
connection = {"sslmode": "require", "target_session_attrs": "read-write"}
postgres_credentials = {
    "host": os.getenv("DB_DESTINATION_HOST"),
    "port": os.getenv("DB_DESTINATION_PORT"),
    "dbname": os.getenv("DB_DESTINATION_NAME"),
    "user": os.getenv("DB_DESTINATION_USER"),
    "password": os.getenv("DB_DESTINATION_PASSWORD"),
}

connection.update(postgres_credentials)

with psycopg.connect(**connection) as conn:

    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")
        data = cur.fetchall()
        columns = [col[0] for col in cur.description]

df = pd.DataFrame(data, columns=columns)

In [34]:
df.head()

,id,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,...,device_protection,tech_support,streaming_tv,streaming_movies,gender,senior_citizen,partner,dependents,multiple_lines,target
0,2133,3023-GFLBR,2017-03-01,2019-12-01,Month-to-month,No,Credit card (automatic),86.15,2745.70,Fiber optic,...,No,No,No,Yes,Female,0,Yes,Yes,Yes,1
1,837,0727-BMPLR,2015-04-01,2019-11-01,One year,Yes,Electronic check,100.00,5509.30,Fiber optic,...,Yes,No,Yes,Yes,Female,1,No,No,Yes,1
2,890,9227-LUNBG,2019-10-01,2019-11-01,Month-to-month,No,Electronic check,24.60,24.60,DSL,...,No,No,No,No,Female,0,No,No,No,1
3,1001,7047-YXDMZ,2018-05-01,2019-11-01,Month-to-month,No,Mailed check,20.00,417.70,Fiber optic,...,No,No,No,No,Male,0,No,No,No,0
4,1002,2858-EIMXH,2015-09-01,2019-11-01,One year,Yes,Credit card (automatic),95.85,5016.25,Fiber optic,...,No,No,Yes,Yes,Female,1,Yes,No,Yes,0


In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7019 entries, 0 to 7018
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   id                 7019 non-null   int64         
 1   customer_id        7019 non-null   object        
 2   begin_date         7019 non-null   datetime64[ns]
 3   end_date           7019 non-null   datetime64[ns]
 4   type               7019 non-null   object        
 5   paperless_billing  7019 non-null   object        
 6   payment_method     7019 non-null   object        
 7   monthly_charges    7019 non-null   float64       
 8   total_charges      7019 non-null   float64       
 9   internet_service   7019 non-null   object        
 10  online_security    7019 non-null   object        
 11  online_backup      7019 non-null   object        
 12  device_protection  7019 non-null   object        
 13  tech_support       7019 non-null   object        
 14  streamin

In [36]:
obj_df = df.select_dtypes(include="object")

In [37]:
# определение категориальных колонок, которые будут преобразованы
cat_columns = ["type", "payment_method", "internet_service", "gender"]

encoder_oh = OneHotEncoder(categories='auto', handle_unknown='ignore', 
                           max_categories=10, sparse_output=False, drop='first')

# применение OneHotEncoder к данным. Преобразование категориальных данных в массив
encoded_features = encoder_oh.fit_transform(df[cat_columns].to_numpy())

# преобразование полученных признаков в DataFrame и установка названий колонок
encoded_df = pd.DataFrame(encoded_features, columns=encoder_oh.get_feature_names_out(cat_columns))

# конкатенация исходного DataFrame с новым DataFrame, содержащим закодированные категориальные признаки
obj_df = pd.concat([obj_df, encoded_df], axis=1)

obj_df.head(2)

,customer_id,type,paperless_billing,payment_method,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,...,partner,dependents,multiple_lines,type_One year,type_Two year,payment_method_Credit card (automatic),payment_method_Electronic check,payment_method_Mailed check,internet_service_Fiber optic,gender_Male
0,3023-GFLBR,Month-to-month,No,Credit card (automatic),Fiber optic,No,No,No,No,No,...,Yes,Yes,Yes,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1,0727-BMPLR,One year,Yes,Electronic check,Fiber optic,No,No,Yes,No,Yes,...,No,No,Yes,1.0,0.0,0.0,1.0,0.0,1.0,0.0


In [38]:
num_columns = ["monthly_charges", "total_charges"]
num_df = df.select_dtypes(include='float')

n_knots = 3
degree_spline = 4
n_quantiles=100
degree = 3
n_bins = 5
encode = 'ordinal'
strategy = 'uniform'
subsample = None


# 1. SplineTransformer
encoder_spl = SplineTransformer(n_knots=n_knots, degree=degree_spline)
encoded_features = encoder_spl.fit_transform(df[num_columns].to_numpy())

encoded_df = pd.DataFrame(
    encoded_features, 
    columns=encoder_spl.get_feature_names_out(num_columns)
)
# encoded_df.columns = [encoded_df.columns[1 + len(num_columns):]]
num_df = pd.concat([num_df, encoded_df], axis=1)


# 2. QuantileTransformer
encoder_q = QuantileTransformer(n_quantiles=n_quantiles)
encoded_features = encoder_q.fit_transform(df[num_columns].to_numpy())

# Преобразование в DataFrame
encoded_df = pd.DataFrame(encoded_features, columns=encoder_q.get_feature_names_out(num_columns))
encoded_df.columns = [col + f"_q_{n_quantiles}" for col in num_columns]
num_df = pd.concat([num_df, encoded_df], axis=1)


# 3. RobustScaler
encoder_rb = RobustScaler()
encoded_features = encoder_rb.fit_transform(df[num_columns].to_numpy())

# Преобразование в DataFrame
encoded_df = pd.DataFrame(encoded_features, columns=encoder_rb.get_feature_names_out(num_columns))
encoded_df.columns = [col + f"_robust" for col in num_columns]
num_df = pd.concat([num_df, encoded_df], axis=1)


# 4. PolynomialFeatures
encoder_pol = PolynomialFeatures(degree=degree)
encoded_features = encoder_pol.fit_transform(df[num_columns].to_numpy())

# Преобразование в DataFrame
encoded_df = pd.DataFrame(encoded_features, columns=encoder_pol.get_feature_names_out(num_columns))
encoded_df.columns = [f"poly_{i}" for i in range(encoded_df.shape[1])]
num_df = pd.concat([num_df, encoded_df], axis=1)

# 5. KBinsDiscretizer
encoder_kbd = KBinsDiscretizer(n_bins=n_bins, encode=encode, strategy=strategy, subsample=subsample)
encoded_features = encoder_kbd.fit_transform(df[num_columns].to_numpy())

# Преобразование в DataFrame
encoded_df = pd.DataFrame(encoded_features, columns=encoder_kbd.get_feature_names_out(num_columns))
encoded_df.columns = [col + f'_bin' for col in num_columns]
num_df = pd.concat([num_df, encoded_df], axis=1)


num_df.head(2)

,monthly_charges,total_charges,monthly_charges_sp_0,monthly_charges_sp_1,monthly_charges_sp_2,monthly_charges_sp_3,monthly_charges_sp_4,monthly_charges_sp_5,total_charges_sp_0,total_charges_sp_1,...,poly_2,poly_3,poly_4,poly_5,poly_6,poly_7,poly_8,poly_9,monthly_charges_bin,total_charges_bin
0,86.15,2745.7,0.0,0.007381,0.270998,0.585250,0.135736,0.000634,0.000787,0.143135,...,2745.7,7421.8225,236542.055,7538868.49,639390.008375,2.037810e+07,6.494735e+08,2.069947e+10,3.0,1.0
1,100.00,5509.3,0.0,0.000808,0.144091,0.588964,0.259704,0.006434,0.000000,0.012019,...,5509.3,10000.0000,550930.000,30352386.49,1000000.000000,5.509300e+07,3.035239e+09,1.672204e+11,4.0,3.0


In [39]:
num_columns = ["monthly_charges", "total_charges"]

In [40]:
# Преобразование для числовых колонок
numeric_transformer = ColumnTransformer(transformers=[
    ('spl', encoder_spl, num_columns),
    ('q', encoder_q, num_columns),
    ('rb', encoder_rb, num_columns),
    ('pol', encoder_pol, num_columns),
    ('kbd', encoder_kbd, num_columns)
    ]
)

categorical_transformer = Pipeline(steps=[('encoder', encoder_oh)])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_columns),
    ('cat', categorical_transformer, cat_columns)
],
n_jobs=-1)

# Применение преобразований
encoded_features = preprocessor.fit_transform(df)

# Создание DataFrame для закодированных фич
encoded_df = pd.DataFrame(encoded_features)

# Объединение с исходным DataFrame
transformed_df = pd.DataFrame(encoded_features, columns=preprocessor.get_feature_names_out())

# Назначение на переменную df
df = pd.concat([df, transformed_df], axis=1)
df.head(2)

,id,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,...,num__pol__total_charges^3,num__kbd__monthly_charges,num__kbd__total_charges,cat__type_One year,cat__type_Two year,cat__payment_method_Credit card (automatic),cat__payment_method_Electronic check,cat__payment_method_Mailed check,cat__internet_service_Fiber optic,cat__gender_Male
0,2133,3023-GFLBR,2017-03-01,2019-12-01,Month-to-month,No,Credit card (automatic),86.15,2745.7,Fiber optic,...,2.069947e+10,3.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1,837,0727-BMPLR,2015-04-01,2019-11-01,One year,Yes,Electronic check,100.00,5509.3,Fiber optic,...,1.672204e+11,4.0,3.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0


In [41]:
df.isna().sum()

id                                             0
customer_id                                    0
begin_date                                     0
end_date                                       0
type                                           0
paperless_billing                              0
payment_method                                 0
monthly_charges                                0
total_charges                                  0
internet_service                               0
online_security                                0
online_backup                                  0
device_protection                              0
tech_support                                   0
streaming_tv                                   0
streaming_movies                               0
gender                                         0
senior_citizen                                 0
partner                                        0
dependents                                     0
multiple_lines      

In [42]:
preprocessor

ColumnTransformer(n_jobs=-1,
                  transformers=[('num',
                                 ColumnTransformer(transformers=[('spl',
                                                                  SplineTransformer(degree=4,
                                                                                    n_knots=3),
                                                                  ['monthly_charges',
                                                                   'total_charges']),
                                                                 ('q',
                                                                  QuantileTransformer(n_quantiles=100),
                                                                  ['monthly_charges',
                                                                   'total_charges']),
                                                                 ('rb',
                                                                  RobustScaler(),
                                                                  ['monthly_charges',
                                                                   'total_charges']),
                                                                 ('pol',
                                                                  PolynomialFeatures(degree=3),
                                                                  ['monthly_char...
                                                                   'total_charges']),
                                                                 ('kbd',
                                                                  KBinsDiscretizer(encode='ordinal',
                                                                                   strategy='uniform',
                                                                                   subsample=None),
                                                                  ['monthly_charges',
                                                                   'total_charges'])]),
                                 ['monthly_charges', 'total_charges']),
                                ('cat',
                                 Pipeline(steps=[('encoder',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                max_categories=10,
                                                                sparse_output=False))]),
                                 ['type', 'payment_method', 'internet_service',
                                  'gender'])])

In [43]:
os.environ["MLFLOW_S3_ENDPOINT_URL"] = 'https://storage.yandexcloud.net'
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv('AWS_ACCESS_KEY_ID')
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv('AWS_SECRET_ACCESS_KEY')

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    mlflow.sklearn.log_model(preprocessor, "column_transformer")

2024/11/08 08:41:21 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


In [44]:
run_id

'98c05191cd0f4221ab82bd4206b4e014'

In [45]:
df.columns

Index(['id', 'customer_id', 'begin_date', 'end_date', 'type',
       'paperless_billing', 'payment_method', 'monthly_charges',
       'total_charges', 'internet_service', 'online_security', 'online_backup',
       'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies',
       'gender', 'senior_citizen', 'partner', 'dependents', 'multiple_lines',
       'target', 'num__spl__monthly_charges_sp_0',
       'num__spl__monthly_charges_sp_1', 'num__spl__monthly_charges_sp_2',
       'num__spl__monthly_charges_sp_3', 'num__spl__monthly_charges_sp_4',
       'num__spl__monthly_charges_sp_5', 'num__spl__total_charges_sp_0',
       'num__spl__total_charges_sp_1', 'num__spl__total_charges_sp_2',
       'num__spl__total_charges_sp_3', 'num__spl__total_charges_sp_4',
       'num__spl__total_charges_sp_5', 'num__q__monthly_charges',
       'num__q__total_charges', 'num__rb__monthly_charges',
       'num__rb__total_charges', 'num__pol__1', 'num__pol__monthly_charges',
       'num__pol

In [46]:
connection = {"sslmode": "require", "target_session_attrs": "read-write"}
postgres_credentials = {
    "host": os.getenv("DB_DESTINATION_HOST"),
    "port": os.getenv("DB_DESTINATION_PORT"),
    "dbname": os.getenv("DB_DESTINATION_NAME"),
    "user": os.getenv("DB_DESTINATION_USER"),
    "password": os.getenv("DB_DESTINATION_PASSWORD"),
}

connection.update(postgres_credentials)

with psycopg.connect(**connection) as conn:

    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")
        data = cur.fetchall()
        columns = [col[0] for col in cur.description]

df = pd.DataFrame(data, columns=columns)

In [53]:
X = df.drop(columns=['target', 'id', 'customer_id'])  # Замените 'target_column' на название вашей целевой переменной
y = df['target']

# Разделение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = CatBoostClassifier(iterations=1000, learning_rate=0.1, verbose=0)

full_pipe = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', model)
])

In [54]:
full_pipe.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(n_jobs=-1,
                                   transformers=[('num',
                                                  ColumnTransformer(transformers=[('spl',
                                                                                   SplineTransformer(degree=4,
                                                                                                     n_knots=3),
                                                                                   ['monthly_charges',
                                                                                    'total_charges']),
                                                                                  ('q',
                                                                                   QuantileTransformer(n_quantiles=100),
                                                                                   ['monthly_charges',
                                                                                    'total_charges']),
                                                                                  ('rb',
                                                                                   RobustScaler(),
                                                                                   ['monthly_charges',
                                                                                    'total_charges']),
                                                                                  ('pol',
                                                                                   PolynomialFeatures(...
                                                                                                    strategy='uniform',
                                                                                                    subsample=None),
                                                                                   ['monthly_charges',
                                                                                    'total_charges'])]),
                                                  ['monthly_charges',
                                                   'total_charges']),
                                                 ('cat',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore',
                                                                                 max_categories=10,
                                                                                 sparse_output=False))]),
                                                  ['type', 'payment_method',
                                                   'internet_service',
                                                   'gender'])])),
                ('model',
                 <catboost.core.CatBoostClassifier object at 0x7f7593912800>)])

In [56]:
# Предсказанные значения и вероятности
proba = full_pipe.predict_proba(X_test)[:, 1]
prediction = full_pipe.predict(X_test)

# Истинные метки и данные для предсказания
y_true = y_test

# Заведите словарь со всеми метриками
metrics = {}

# Посчитайте метрики из модуля sklearn.metrics с нормализацией
_, err1, _, err2 = confusion_matrix(y_test, prediction, normalize='all').ravel()

auc = roc_auc_score(y_true, proba)
precision = precision_score(y_true, prediction)
recall = recall_score(y_true, prediction)
f1 = f1_score(y_true, prediction)
logloss = log_loss(y_true, proba)

# Запишите значения метрик в словарь 
metrics["err1"] = err1
metrics["err2"] = err2
metrics["auc"] = auc
metrics["precision"] = precision
metrics["recall"] = recall
metrics["f1"] = f1
metrics["logloss"] = logloss

In [59]:
model = full_pipe.named_steps['model']

In [74]:
EXPERIMENT_NAME = 'churn_laptev_ilya_sergeevich_2'
RUN_NAME = 'cb_feature_engineering'
REGISTRY_MODEL_NAME = 'cb_with_features_info_version_id_etc'

In [69]:
pip_requirements = '../requirements.txt'
signature = mlflow.models.infer_signature(X_test, prediction)
input_example = X_test[:10]
metadata = {'model_type': 'monthly'}

/home/mle-user/mle-mlflow/.venv_mle_mlflow/lib/python3.10/site-packages/mlflow/models/signature.py:212: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  inputs = _infer_schema(model_input) if model_input is not None else None


In [70]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id

In [ ]:
with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    mlflow.log_metrics(metrics)
    # ваш код здесь
    model_info = mlflow.catboost.log_model(cb_model=model,
                                          metadata=metadata,
                                          artifact_path='models',
                                          signature=signature,
                                          pip_requirements=pip_requirements,
                                          input_example=input_example,
                                          registered_model_name=REGISTRY_MODEL_NAME,
                                          await_registration_for=60)

Successfully registered model 'cb_with_features_info_version_id_etc'.
2024/11/08 09:06:39 INFO mlflow.tracking._model_registry.client: Waiting up to 60 seconds for model version to finish creation. Model name: cb_with_features_info_version_id_etc, version 1
Created version '1' of model 'cb_with_features_info_version_id_etc'.


AttributeError: 'ModelInfo' object has no attribute 'version'

In [77]:
# Извлечение информации о зарегистрированной модели
# model_version_id = model_info.version  # номер зарегистрированной модели
# model_registered_name = model_info.registered_model_name  # название зарегистрированной модели
run_id = model_info.run_id  # run_id, в рамках которого была зарегистрирована модель

In [78]:
run_id 

'd5edc21c85df45e582cbfca3c2eb65b0'